# Rodent AI — FOMO Training (MobileNetV2 0.35) on Google Colab

Trains a **FOMO** model — same architecture as Edge Impulse:
- Input: **96×96 grayscale**
- Backbone: **MobileNetV2 α=0.35** (pretrained ImageNet, fine-tuned)
- Output: **12×12 grid** (stride 8), each cell predicts rat / background
- Export: **TFLite int8**

## Before running
1. `fomo_dataset.zip` must be uploaded to your **Google Drive**
   (built from **all** of `ei_curated`: 4192 train + 1040 test — `cur_` rats plus
   `bg_`/`neg_`/`tgt_` backgrounds).
2. Runtime → Change runtime type → **T4 GPU**.
3. Run all cells in order.

## 1. Install dependencies

In [1]:
!pip install -q tensorflow Pillow numpy
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow


ModuleNotFoundError: No module named 'tensorflow'

## 2. Mount Google Drive and unzip dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile

# ── EDIT this path if your zip is in a sub-folder ──────────────────────────
ZIP_PATH = '/content/drive/MyDrive/fomo_dataset.zip'
DEST     = '/content/fomo_dataset'
# ───────────────────────────────────────────────────────────────────────────

if not os.path.exists(DEST):
    print('Unzipping...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DEST)
    print('Done.')
else:
    print('Already unzipped.')

# Quick sanity check
for split in ('training', 'testing'):
    imgs = [f for f in os.listdir(f'{DEST}/{split}') if f.endswith('.jpg')]
    print(f'{split}: {len(imgs)} images')

## 3. Parse labels → FOMO 12×12 grid targets

Each image maps to a **12×12×1** float label. Each rat marks the cell its centroid falls in **plus a 1-cell border** (a 3×3 block, `LABEL_SPREAD`) — this *label spreading* gives the network up to ~9× more positive signal than a single cell, lifting recall and calibrating the scores. Background images are all zeros.


In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image

IMG_SIZE     = 96    # FOMO input (96×96 px)
GRID_SIZE    = 12    # 96 / stride(8) = 12
STRIDE       = IMG_SIZE // GRID_SIZE   # 8
LABEL_SPREAD = 1     # dilate each rat centroid by this many cells (1 = 3×3 block)

def boxes_to_fomo_label(boxes_info, orig_w, orig_h):
    """EI box list [{x,y,width,height}] → (12,12,1) float32 label grid.

    Label spreading: each rat marks its centroid cell PLUS a LABEL_SPREAD-cell
    border around it (LABEL_SPREAD=1 → a 3×3 block), clamped to the grid. A
    single centroid cell is far too sparse (~1% of 144 cells) — the extra
    positive cells give the network up to ~9× more rat signal per image, which
    lifts recall and calibrates the output scores. On-device you recover the
    point detection by taking the centroid of each above-threshold blob.
    """
    label = np.zeros((GRID_SIZE, GRID_SIZE, 1), dtype=np.float32)
    for b in boxes_info:
        cx = b['x'] + b['width']  / 2.0
        cy = b['y'] + b['height'] / 2.0
        # Scale centroid to 96×96, then to a grid cell
        cx96 = cx * IMG_SIZE / orig_w
        cy96 = cy * IMG_SIZE / orig_h
        col = int(min(cx96 // STRIDE, GRID_SIZE - 1))
        row = int(min(cy96 // STRIDE, GRID_SIZE - 1))
        # Dilate the centroid into a (2*LABEL_SPREAD+1)² block, clamped to grid
        r0 = max(0, row - LABEL_SPREAD)
        r1 = min(GRID_SIZE - 1, row + LABEL_SPREAD)
        c0 = max(0, col - LABEL_SPREAD)
        c1 = min(GRID_SIZE - 1, col + LABEL_SPREAD)
        label[r0:r1 + 1, c0:c1 + 1, 0] = 1.0
    return label

def load_split(split_dir):
    split_dir = Path(split_dir)
    data = json.loads((split_dir / 'bounding_boxes.labels').read_text())
    bb = data['boundingBoxes']
    images, labels = [], []
    skipped = 0
    for fname, boxes_info in bb.items():
        img_path = split_dir / fname
        if not img_path.exists():
            skipped += 1
            continue
        try:
            im = Image.open(img_path).convert('L')   # grayscale
            orig_w, orig_h = im.size
            im = im.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            arr = np.array(im, dtype=np.float32) / 255.0
            arr = arr[..., np.newaxis]               # (96, 96, 1)
        except Exception:
            skipped += 1
            continue
        label = boxes_to_fomo_label(boxes_info, orig_w, orig_h)
        images.append(arr)
        labels.append(label)
    if skipped:
        print(f'  [skipped {skipped} missing/corrupt files]')
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.float32)

print('Loading training set...')
X_train, Y_train = load_split(f'{DEST}/training')
print(f'  X={X_train.shape}  Y={Y_train.shape}')

print('Loading test set...')
X_test, Y_test = load_split(f'{DEST}/testing')
print(f'  X={X_test.shape}  Y={Y_test.shape}')

n_pos = int((Y_train.sum(axis=(1, 2, 3)) > 0).sum())
print(f'\nTrain: {n_pos} positives (rat), {len(Y_train)-n_pos} backgrounds')
print(f'Test:  {int((Y_test.sum(axis=(1,2,3))>0).sum())} positives, '
      f'{int((Y_test.sum(axis=(1,2,3))==0).sum())} backgrounds')
print(f'Positive-cell fraction (LABEL_SPREAD={LABEL_SPREAD}): {Y_train.mean():.4f} '
      f'(~0.011 at single-cell)')


## 4. Build FOMO model (MobileNetV2 α=0.35)

Exact same logic as Edge Impulse FOMO:
- MobileNetV2 truncated at `block_6_expand_relu` → **12×12×96** feature map (stride 8)
- Extra 1×1 Conv (32 ch, ReLU) + 1×1 Conv (1 ch, sigmoid) → **12×12×1** detection grid

In [ ]:
import math
import tensorflow as tf
from tensorflow.keras import layers, Model

def build_fomo(input_shape=(96, 96, 1), alpha=0.35, pos_frac=0.05):
    inp = layers.Input(shape=input_shape, name='input_grayscale')

    # MobileNetV2 expects 3 channels — repeat the grayscale channel 3×.
    # Use a native Concatenate (NOT Lambda): Keras 3 can't infer a Lambda's
    # output shape here (raises NotImplementedError), and Lambda+tf.repeat is
    # also fragile for the int8 TFLite export later. Concatenate is both.
    x = layers.Concatenate(axis=-1, name='gray_to_rgb')([inp, inp, inp])

    # MobileNetV2's ImageNet weights were trained on inputs scaled to [-1, 1],
    # but our images arrive as [0, 1]. Without this remap the pretrained
    # features get the wrong range and barely activate — a second cause of the
    # collapse. Rescaling is a plain affine op, fully int8-TFLite safe.
    x = layers.Rescaling(scale=2.0, offset=-1.0, name='to_pm1')(x)  # [0,1] -> [-1,1]

    # MobileNetV2 0.35 backbone (ImageNet weights)
    base = tf.keras.applications.MobileNetV2(
        input_shape=(input_shape[0], input_shape[1], 3),
        alpha=alpha,
        include_top=False,
        weights='imagenet'
    )

    # Find stride-8 feature layer (block_6 → 12×12 for 96×96 input)
    target_layer = None
    for lyr in base.layers:
        if 'block_6_expand_relu' in lyr.name:
            target_layer = lyr.name
            break
    # Fallback: first layer whose spatial output is 12×12
    if target_layer is None:
        for lyr in base.layers:
            out = lyr.output
            if hasattr(out, 'shape') and len(out.shape) == 4 and out.shape[1] == 12:
                target_layer = lyr.name
                break
    print(f'Backbone truncated at: {target_layer}')

    backbone = Model(
        inputs=base.input,
        outputs=base.get_layer(target_layer).output,
        name='mobilenetv2_035'
    )
    print(f'Feature map shape: {backbone.output_shape}')

    x = backbone(x)                                     # (B, 12, 12, C)

    # FOMO head
    x = layers.Conv2D(32, 1, padding='same',
                      activation='relu',    name='head_conv')(x)
    # Prior-bias init (RetinaNet trick): start the sigmoid near the true rat-cell
    # frequency (pos_frac) instead of 50%, so the background gradient doesn't
    # swamp training and collapse the model to all-empty. pos_frac is measured
    # from the (spread) labels — see the build call below — so it auto-adapts to
    # whatever LABEL_SPREAD is set to.
    pos_frac = min(max(pos_frac, 1e-4), 0.5)
    output_bias = tf.keras.initializers.Constant(-math.log((1.0 - pos_frac) / pos_frac))
    x = layers.Conv2D(1,  1, padding='same',
                      activation='sigmoid', bias_initializer=output_bias,
                      name='fomo_output')(x)  # (B, 12, 12, 1)

    return Model(inputs=inp, outputs=x, name='FOMO_rat')

# Prior bias from the actual positive-cell fraction of the (spread) labels.
POS_FRAC = float(Y_train.mean())
print(f'Output prior set from positive-cell fraction: {POS_FRAC:.4f}')
model = build_fomo(pos_frac=POS_FRAC)
model.summary(line_length=80)


## 5. Loss function and compile

Binary **focal loss** (γ=2, α=0.5) handles the severe imbalance — only ~1–4 positive cells out of 144 per image. α is set to 0.5 (up from the RetinaNet default 0.25) to give the rare rat cells more weight.


In [ ]:
def binary_focal_loss(gamma=2.0, alpha=0.5):
    """Focal loss — suppresses easy negatives, focuses on hard positives.

    alpha is the weight given to POSITIVE (rat) cells; (1-alpha) to background.
    Bumped 0.25 -> 0.5 to up-weight the rare rat cells: with only ~1% positives
    the background term otherwise dominates the gradient and the model collapses
    to predicting all-empty. Raise toward 0.75 if recall is still too low.
    """
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        at = tf.where(tf.equal(y_true, 1),
                      tf.ones_like(y_true) * alpha,
                      tf.ones_like(y_true) * (1 - alpha))
        return -tf.reduce_mean(at * tf.pow(1.0 - pt, gamma) * tf.math.log(pt))
    return loss

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=binary_focal_loss(gamma=2.0, alpha=0.5),
    metrics=['binary_accuracy']
)
print('Model compiled.')


## 6. Train

In [ ]:
EPOCHS    = 60
BATCH     = 32
SAVE_PATH = '/content/fomo_best.keras'

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        SAVE_PATH, save_best_only=True, monitor='val_loss', verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1
    ),
]

history = model.fit(
    X_train, Y_train,
    validation_data=(X_test, Y_test),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks,
    verbose=1
)

## 7. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'],     label='train')
ax1.plot(history.history['val_loss'], label='val')
ax1.set_title('Focal Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['binary_accuracy'],     label='train')
ax2.plot(history.history['val_binary_accuracy'], label='val')
ax2.set_title('Binary Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=120)
plt.show()

## 8. Evaluate — Precision / Recall / F1 on test set

In [ ]:
CONF_THRESHOLD = 0.3   # FOMO cell is "rat" if score >= this
# Lowered 0.5 -> 0.3: FOMO scores tend to sit low (few positive cells, focal
# loss + prior bias keep outputs conservative). Sweep the values below to pick
# the threshold with the best F1 for your deployment.

preds = model.predict(X_test, batch_size=32, verbose=0)  # (N, 12, 12, 1)
pred_bin = (preds >= CONF_THRESHOLD).astype(np.float32)

tp = int(np.sum((pred_bin == 1) & (Y_test == 1)))
fp = int(np.sum((pred_bin == 1) & (Y_test == 0)))
fn = int(np.sum((pred_bin == 0) & (Y_test == 1)))

precision = tp / (tp + fp + 1e-8)
recall    = tp / (tp + fn + 1e-8)
f1        = 2 * precision * recall / (precision + recall + 1e-8)

print(f'Threshold : {CONF_THRESHOLD}')
print(f'TP={tp}  FP={fp}  FN={fn}')
print(f'Precision : {precision:.3f}')
print(f'Recall    : {recall:.3f}')
print(f'F1        : {f1:.3f}')

# Per-image false positive rate (images with ≥1 FP cell, ground truth = background)
bg_mask = Y_test.sum(axis=(1, 2, 3)) == 0
fp_images = int((pred_bin[bg_mask].sum(axis=(1, 2, 3)) > 0).sum())
print(f'\nFalse-positive images: {fp_images} / {int(bg_mask.sum())} backgrounds')

# --- Threshold sweep: find the best F1 without re-running the model ----------
print('\nthr    P      R      F1')
for thr in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
    pb = (preds >= thr).astype(np.float32)
    _tp = float(np.sum((pb == 1) & (Y_test == 1)))
    _fp = float(np.sum((pb == 1) & (Y_test == 0)))
    _fn = float(np.sum((pb == 0) & (Y_test == 1)))
    _p = _tp / (_tp + _fp + 1e-8)
    _r = _tp / (_tp + _fn + 1e-8)
    _f = 2 * _p * _r / (_p + _r + 1e-8)
    print(f'{thr:.1f}  {_p:.3f}  {_r:.3f}  {_f:.3f}')


## 9. Export TFLite int8

Quantises weights **and** activations to int8 using 100 representative training images for calibration.

In [ ]:
# Restore best weights
model.load_weights(SAVE_PATH)

# Representative dataset for int8 calibration
def representative_dataset():
    indices = np.random.choice(len(X_train), size=min(100, len(X_train)), replace=False)
    for i in indices:
        yield [X_train[i:i+1]]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations         = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

TFLITE_PATH = '/content/fomo_rat_int8.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

print(f'TFLite int8: {TFLITE_PATH}  ({len(tflite_model)//1024} KB)')

## 10. Save to Google Drive + download

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

drive_out = Path('/content/drive/MyDrive/rodent_fomo_model')
drive_out.mkdir(exist_ok=True)

shutil.copy(SAVE_PATH,                     drive_out / 'fomo_best.keras')
shutil.copy(TFLITE_PATH,                   drive_out / 'fomo_rat_int8.tflite')
shutil.copy('/content/training_curves.png', drive_out / 'training_curves.png')

print('Saved to Drive:', drive_out)

# Direct download to your computer
files.download(TFLITE_PATH)

## Notes on ESP32-CAM deployment

The exported `fomo_rat_int8.tflite` is a **custom model** (not EI-wrapped).

### Option A — Re-import into Edge Impulse *(easiest, no firmware change)*
Upload the `.tflite` via *Deployment → Bring Your Own Model* (Enterprise plan).
EI wraps it into an Arduino library → same `rodent_webserver.ino` firmware, unchanged.

### Option B — TFLite Micro directly *(no EI required)*
Replace the EI SDK with the ESP32 TFLite Micro library.
Parse the output tensor yourself (12×12×1 float after dequantisation).
Ask your assistant to generate the updated firmware once you have the `.tflite`.

---

### How the FOMO output maps to detections
- Output tensor: **12×12×1** (int8, dequantise with scale+zero_point from tensor metadata)
- For each cell `(row, col)` with score ≥ threshold: rat centroid = `(col*8+4, row*8+4)` pixels in 96×96 space
- Scale back to original frame if needed